# CliffForge — Evaluation on Dablander MMP Dataset (12,594 pairs)

**Goal**: Evaluate the CliffForge Siamese relative-affinity model on the full published
Moonshot MMP dataset (1,924 molecules, 12,594 pairs) from:
> Dablander et al., *QSAR-activity-cliff-experiments*  
> https://github.com/MarkusFerdinandDablander/QSAR-activity-cliff-experiments

**What this notebook does (end-to-end):**
1. Install all dependencies (compatible pinned versions)
2. Clone the CliffForge repository
3. Download both Dablander CSV files directly from GitHub
4. Pre-process: canonicalize SMILES (stereo-free), convert IC50→pIC50, build scaffold-aware split
5. Train a 3-seed Siamese D-MPNN ensemble on the MMP training pairs
6. Evaluate: direction accuracy, within-series Spearman ρ, binary cliff detection (AP, F1, ROC-AUC) with cluster-bootstrap CIs
7. Compare against a 5-NN similarity-weighted baseline
8. Save all results to `results_mmp/`

> **Runtime estimate (Colab T4 GPU):** ~25–35 min total

> ⚠️ **Enable GPU first**: *Runtime → Change runtime type → T4 GPU*

## 0 · Environment Check

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU found:', result.stdout.strip())
else:
    print('No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')
print('Python:', sys.version)

## 1 · Install Dependencies

> Takes ~4–5 min on a fresh Colab instance.

In [ ]:
%%capture
# Core scientific stack
!pip install -q numpy==1.23.5 scipy==1.9.3 pandas==1.5.3 scikit-learn==1.1.3

# RDKit
!pip install -q rdkit-pypi==2022.9.5

# PyTorch (CUDA 11.3 wheel — matches Colab T4)
!pip install -q torch==1.12.1+cu113 torchvision==0.13.1+cu113 \
    --extra-index-url https://download.pytorch.org/whl/cu113

# PyTorch Geometric (must match torch version)
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    torch-geometric==2.3.1 \
    -f https://data.pyg.org/whl/torch-1.12.1+cu113.html

# ChemProp baseline
!pip install -q chemprop==1.6.1

print('Installation complete.')

In [ ]:
import importlib
for pkg in ['numpy', 'scipy', 'pandas', 'sklearn', 'rdkit', 'torch', 'torch_geometric']:
    try:
        m = importlib.import_module(pkg)
        print(f'  OK {pkg} {getattr(m, "__version__", "?")}')
    except ImportError:
        print(f'  MISSING {pkg}')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2 · Clone CliffForge Repository

In [ ]:
import os, sys

REPO = '/content/TELEMEDICON26'
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/chidwipak/TELEMEDICON26.git {REPO}
else:
    print('Repo already cloned.')

src_path = os.path.join(REPO, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import cliffforge
print('cliffforge imported from:', cliffforge.__file__)

## 3 · Download & Pre-Process Dablander MMP Dataset

Two files:
- `molecule_data_clean.csv` — 1,924 molecules with `SMILES` and `f_avg_IC50 [uM]`
- `MMP_data_clean.csv` — 12,594 MMP pairs with `smiles_1`, `smiles_2`

Steps:
1. Canonicalize SMILES (stereo-free, same as original `data_prep.py`)
2. Convert IC50 [µM] → pIC50 = 6 − log₁₀(IC50)
3. Scaffold-aware train/test split (Bemis-Murcko, ≥8 members, 35% test)
4. Filter MMP pairs to train-train pairs for training

In [ ]:
import os
os.makedirs('/content/mmp_data/raw', exist_ok=True)

BASE = 'https://raw.githubusercontent.com/MarkusFerdinandDablander/QSAR-activity-cliff-experiments/main/data/postera_sars_cov_2_mpro'
!wget -q -O /content/mmp_data/raw/molecule_data_clean.csv  {BASE}/molecule_data_clean.csv
!wget -q -O /content/mmp_data/raw/MMP_data_clean.csv       {BASE}/MMP_data_clean.csv

import pandas as pd
mol_raw = pd.read_csv('/content/mmp_data/raw/molecule_data_clean.csv')
mmp_raw = pd.read_csv('/content/mmp_data/raw/MMP_data_clean.csv')

print(f'Molecules: {len(mol_raw):,}  |  MMP pairs: {len(mmp_raw):,}')
print('Molecule columns:', list(mol_raw.columns))
print('MMP columns     :', list(mmp_raw.columns))

In [ ]:
import math
import numpy as np
from collections import defaultdict
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')

def canon2d(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return None, None
    flat = Chem.MolToSmiles(m, isomericSmiles=False)
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(flat))
        scaf_smi = Chem.MolToSmiles(scaf)
    except Exception:
        scaf_smi = 'NONE'
    return flat, scaf_smi

IC50_COL = 'f_avg_IC50 [uM]'
records = []
for _, row in mol_raw.iterrows():
    ic50 = float(row[IC50_COL])
    if ic50 <= 0:
        continue
    flat, scaf = canon2d(row['SMILES'])
    if flat is None:
        continue
    records.append({'smiles': flat, 'pIC50': 6.0 - math.log10(ic50), 'scaf': scaf})

# Deduplicate by canonical SMILES
by_smi = defaultdict(list)
for r in records:
    by_smi[r['smiles']].append(r['pIC50'])

deduped = []
for smi, vals in by_smi.items():
    scaf = next(r['scaf'] for r in records if r['smiles'] == smi)
    deduped.append({'smiles': smi, 'pIC50': float(np.mean(vals)), 'scaf': scaf, 'n_rep': len(vals)})

print(f'Raw molecules: {len(mol_raw):,} → after canon+dedup: {len(deduped):,}')

In [ ]:
# Scaffold-aware split — mirrors data_prep.py
MIN_SERIES = 8
TEST_FRAC  = 0.35
SPLIT_SEED = 17

rng = np.random.RandomState(SPLIT_SEED)
scaf_members = defaultdict(list)
for rec in deduped:
    scaf_members[rec['scaf']].append(rec)

for scaf, members in scaf_members.items():
    sz = len(members)
    if sz >= MIN_SERIES:
        idx = rng.permutation(sz)
        n_test = max(3, int(round(sz * TEST_FRAC)))
        n_test = min(n_test, sz - 3)
        test_ids = set(idx[:n_test].tolist())
        for i, rec in enumerate(members):
            rec['series'] = scaf
            rec['series_size'] = sz
            rec['split'] = 'test' if i in test_ids else 'train'
    else:
        for rec in members:
            rec['series'] = scaf if sz > 1 else 'singleton'
            rec['series_size'] = sz
            rec['split'] = 'train'

df_all = pd.DataFrame(deduped)[['smiles', 'pIC50', 'series', 'series_size', 'split', 'n_rep']]
n_train = int((df_all.split == 'train').sum())
n_test  = int((df_all.split == 'test').sum())
print(f'Total: {len(df_all):,}  |  train: {n_train:,}  |  test: {n_test:,}')
print(f'Test series: {df_all[df_all.split=="test"]["series"].nunique()}')

In [ ]:
smi_to_split  = dict(zip(df_all.smiles, df_all.split))
smi_to_pic50  = dict(zip(df_all.smiles, df_all.pIC50))
smi_to_series = dict(zip(df_all.smiles, df_all.series))

def canon_flat(smi):
    m = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(m, isomericSmiles=False) if m else None

print('Canonicalizing and classifying MMP pairs...')
mmp_rows, skipped = [], 0
for _, row in mmp_raw.iterrows():
    s1, s2 = canon_flat(row['smiles_1']), canon_flat(row['smiles_2'])
    if not s1 or not s2 or s1 == s2:
        skipped += 1; continue
    if s1 not in smi_to_split or s2 not in smi_to_split:
        skipped += 1; continue
    sp1, sp2 = smi_to_split[s1], smi_to_split[s2]
    pair_type = ('train_train' if sp1=='train' and sp2=='train' else
                 'test_test'   if sp1=='test'  and sp2=='test'  else 'test_train')
    mmp_rows.append({
        'smiles_1': s1, 'smiles_2': s2,
        'pIC50_1': smi_to_pic50[s1], 'pIC50_2': smi_to_pic50[s2],
        'delta_pIC50': smi_to_pic50[s1] - smi_to_pic50[s2],
        'abs_delta': abs(smi_to_pic50[s1] - smi_to_pic50[s2]),
        'split_1': sp1, 'split_2': sp2, 'pair_type': pair_type,
        'series_1': smi_to_series[s1], 'series_2': smi_to_series[s2],
        'same_series': smi_to_series[s1] == smi_to_series[s2],
        'transformation': row.get('transformation', ''),
    })

df_mmp = pd.DataFrame(mmp_rows)
df_train_pairs = df_mmp[df_mmp.pair_type == 'train_train'].reset_index(drop=True)

print(f'Total MMP pairs: {len(df_mmp):,}  (skipped {skipped:,})')
for pt in ['train_train', 'test_train', 'test_test']:
    print(f'  {pt}: {(df_mmp.pair_type==pt).sum():,}')
print(f'\nTraining pairs: {len(df_train_pairs):,}')

In [ ]:
from rdkit import DataStructs
from rdkit.Chem import AllChem

CLIFF_DELTA = 1.0
CLIFF_SIM   = 0.80

print('Computing Morgan fingerprints...')
smi_list = list(df_all.smiles)
fps_list = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, 2048) for s in smi_list]
smi_to_fp = dict(zip(smi_list, fps_list))

print('Computing Tanimoto for all MMP pairs...')
all_smi1 = list(df_mmp.smiles_1)
all_smi2 = list(df_mmp.smiles_2)
tans = [DataStructs.TanimotoSimilarity(smi_to_fp[s1], smi_to_fp[s2])
        for s1, s2 in zip(all_smi1, all_smi2)]
df_mmp = df_mmp.copy()
df_mmp['tanimoto'] = tans
df_mmp['is_cliff'] = (df_mmp.tanimoto >= CLIFF_SIM) & (df_mmp.abs_delta >= CLIFF_DELTA)

print(f'\nCliff counts (tan>={CLIFF_SIM}, |Δ|>={CLIFF_DELTA}):')
for pt in ['train_train', 'test_train', 'test_test']:
    sub = df_mmp[df_mmp.pair_type == pt]
    n_cl = sub.is_cliff.sum()
    n_close = (sub.tanimoto >= CLIFF_SIM).sum()
    print(f'  {pt:<15}: pairs={len(sub):5,}  close={n_close:4,}  cliffs={n_cl:4,}  prevalence={n_cl/max(len(sub),1):.1%}')

In [ ]:
os.makedirs('/content/mmp_data/processed', exist_ok=True)
df_all.to_csv('/content/mmp_data/processed/all.csv', index=False)
df_all[df_all.split=='train'][['smiles','pIC50']].to_csv('/content/mmp_data/processed/train.csv', index=False)
df_all[df_all.split=='test'] [['smiles','pIC50']].to_csv('/content/mmp_data/processed/test.csv',  index=False)
df_mmp.to_csv('/content/mmp_data/processed/mmp_all.csv', index=False, float_format='%.4f')
df_train_pairs[['smiles_1','smiles_2','delta_pIC50','same_series']].to_csv(
    '/content/mmp_data/processed/mmp_train.csv', index=False, float_format='%.4f')
print('Processed files saved.')

## 4 · Train Siamese D-MPNN on MMP Pairs

Same `SiameseAVES` architecture, but now trained on all **published train-train MMP pairs**
(~10× more training signal than the original within-scaffold TransformDB approach).

Training: SmoothL1 on ΔpIC50, 3 seeds × 70 epochs.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch, Data

from cliffforge.features import mol_to_whole_graph, NODE_DIM, EDGE_DIM
from cliffforge.networks import SiameseAVES

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {DEVICE}')

print('Building graph cache...')
_graph_cache = {}

def get_graph(smi):
    if smi not in _graph_cache:
        mol = Chem.MolFromSmiles(smi)
        g = mol_to_whole_graph(mol)
        if g is None:
            _graph_cache[smi] = None
        else:
            _graph_cache[smi] = Data(
                x=torch.from_numpy(g['x']).float(),
                edge_index=torch.from_numpy(g['edge_index']).long(),
                edge_attr=torch.from_numpy(g['edge_attr']).float()
            )
    return _graph_cache[smi]

for s in df_all.smiles:
    get_graph(s)

n_valid = sum(1 for v in _graph_cache.values() if v is not None)
print(f'Graph cache: {n_valid:,} valid / {len(_graph_cache):,} total')

In [ ]:
class MMPPairDataset(Dataset):
    def __init__(self, df_pairs):
        valid = [(get_graph(r.smiles_1) is not None and get_graph(r.smiles_2) is not None)
                 for r in df_pairs.itertuples()]
        self.pairs = df_pairs[valid].reset_index(drop=True)
        print(f'  MMPPairDataset: {len(self.pairs):,} pairs'
              f' (dropped {sum(not v for v in valid):,} with invalid graphs)')

    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        row = self.pairs.iloc[idx]
        return get_graph(row.smiles_1), get_graph(row.smiles_2), float(row.delta_pIC50)

def mmp_collate(batch):
    return (Batch.from_data_list([b[0] for b in batch]),
            Batch.from_data_list([b[1] for b in batch]),
            torch.tensor([b[2] for b in batch], dtype=torch.float))

print('Building training dataset...')
train_dataset = MMPPairDataset(df_train_pairs)

In [ ]:
import time

MODEL_DIR = '/content/mmp_models'
os.makedirs(MODEL_DIR, exist_ok=True)

def train_seed(seed, epochs=70, batch_size=64):
    torch.manual_seed(seed)
    np.random.seed(seed)
    loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                         collate_fn=mmp_collate, num_workers=0)
    model   = SiameseAVES(NODE_DIM, EDGE_DIM, hidden=128, depth=4, dropout=0.1).to(DEVICE)
    opt     = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    loss_fn = nn.SmoothL1Loss()
    print(f'\nSeed {seed} | pairs={len(train_dataset):,} | batches/ep={len(loader)} | epochs={epochs}')
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        tot, n = 0.0, 0
        for ga, gb, y in loader:
            ga, gb, y = ga.to(DEVICE), gb.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(ga, gb), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tot += float(loss) * len(y); n += len(y)
        if ep % 10 == 0 or ep == epochs - 1:
            print(f'  ep{ep:03d}  loss={tot/n:.4f}  t={time.time()-t0:.0f}s')
    ckpt = os.path.join(MODEL_DIR, f'rel_siamese_mmp_s{seed}.pt')
    torch.save({'state_dict': model.state_dict(), 'seed': seed, 'epochs': epochs,
                'n_train_pairs': len(train_dataset),
                'dataset': 'Dablander MMP train-train pairs'}, ckpt)
    print(f'  Saved → {ckpt}')
    return ckpt

In [ ]:
# EPOCHS = 5   # ← quick smoke-test
EPOCHS = 70  # ← full training

ckpt_paths = [train_seed(seed=s, epochs=EPOCHS) for s in [0, 1, 2]]
print('\nAll seeds trained:', ckpt_paths)

## 5 · Evaluation

In [ ]:
class MMPRelScorer:
    def __init__(self, ckpt_paths):
        self.device = DEVICE
        self.models = []
        for p in ckpt_paths:
            m = SiameseAVES(NODE_DIM, EDGE_DIM, 128, 4, 0.1).to(self.device)
            m.load_state_dict(torch.load(p, map_location=self.device)['state_dict'])
            m.eval()
            self.models.append(m)
        print(f'Loaded {len(self.models)} models.')

    @torch.no_grad()
    def _raw(self, model, ga, gb, bs=256):
        out = []
        for i in range(0, len(ga), bs):
            a = Batch.from_data_list(ga[i:i+bs]).to(self.device)
            b = Batch.from_data_list(gb[i:i+bs]).to(self.device)
            out.append(model(a, b).cpu().numpy())
        return np.concatenate(out) if out else np.zeros(0)

    def pair_delta_all(self, smiles_a, smiles_b):
        ga = [get_graph(s) for s in smiles_a]
        gb = [get_graph(s) for s in smiles_b]
        return np.stack([(self._raw(m,ga,gb) - self._raw(m,gb,ga))/2.0 for m in self.models])

scorer = MMPRelScorer(ckpt_paths)

In [ ]:
print('Building Tanimoto similarity matrix...')
y_all      = df_all.pIC50.values.astype(float)
is_train_a = (df_all.split == 'train').values
fps_all    = [smi_to_fp[s] for s in df_all.smiles]

n_mol = len(df_all)
S = np.zeros((n_mol, n_mol), dtype=np.float32)
for i in range(n_mol):
    S[i] = DataStructs.BulkTanimotoSimilarity(fps_all[i], fps_all)

def knn_predict(S, y, is_train, k=5):
    pred = np.zeros(len(y))
    tr   = np.where(is_train)[0]
    for i in range(len(y)):
        sims = S[i, tr].copy()
        if is_train[i]: sims[tr == i] = -1
        top = np.argsort(-sims)[:k]
        w   = np.clip(sims[top], 1e-6, None)
        pred[i] = float(np.sum(w * y[tr[top]]) / np.sum(w))
    return pred

print('Computing 5-NN predictions...')
knn_pred_all = knn_predict(S, y_all, is_train_a)
print('Done.')

In [ ]:
df_test_pairs = df_mmp[df_mmp.pair_type != 'train_train'].reset_index(drop=True)
valid_mask = [(get_graph(r.smiles_1) is not None and get_graph(r.smiles_2) is not None)
              for r in df_test_pairs.itertuples()]
df_eval = df_test_pairs[valid_mask].reset_index(drop=True)
print(f'Scoring {len(df_eval):,} test-involved pairs...')

pred_all = scorer.pair_delta_all(list(df_eval.smiles_1), list(df_eval.smiles_2))
df_eval = df_eval.copy()
df_eval['pred_rel_mean'] = pred_all.mean(0)
df_eval['pred_rel_sd']   = pred_all.std(0)

smi_to_idx = {s: i for i, s in enumerate(df_all.smiles)}
df_eval['pred_knn_delta'] = [
    knn_pred_all[smi_to_idx[r.smiles_1]] - knn_pred_all[smi_to_idx[r.smiles_2]]
    for r in df_eval.itertuples()
]

os.makedirs('/content/results_mmp', exist_ok=True)
df_eval.to_csv('/content/results_mmp/mmp_test_predictions.csv', index=False, float_format='%.4f')
print('Saved predictions.')

In [ ]:
from scipy.stats import binomtest

def direction_acc(true_d, pred_d):
    m = np.isfinite(pred_d) & (true_d != 0) & (pred_d != 0)
    if m.sum() == 0: return {'n': 0, 'da': None}
    ok = (np.sign(true_d[m]) == np.sign(pred_d[m]))
    rng = np.random.RandomState(0)
    bs = [ok[rng.randint(0, len(ok), len(ok))].mean() for _ in range(2000)]
    p  = float(binomtest(int(ok.sum()), len(ok), 0.5).pvalue)
    return {'n': int(m.sum()), 'da': float(ok.mean()),
            'ci95': [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))],
            'p': p}

dir_results = {}
print(f'{"Subset":<40} {"Model":<15} {"n":>6} {"DirAcc":>8} {"p":>10}')
print('─'*82)
for pt in ['test_test', 'test_train']:
    sub = df_eval[df_eval.pair_type == pt]
    for min_d, lbl in [(0.0,'all'), (0.4,'|d|>=0.4'), (1.0,'cliff(|d|>=1.0)')]:
        s2 = sub[sub.abs_delta >= min_d]
        if len(s2) < 3: continue
        key = f'{pt}|{lbl}'
        dir_results[key] = {}
        for mname, col in [('rel_siamese','pred_rel_mean'), ('knn_5nn','pred_knn_delta')]:
            r = direction_acc(s2.delta_pIC50.values, s2[col].values)
            dir_results[key][mname] = r
            if r['da'] is not None:
                sig = '***' if r['p']<0.001 else ('**' if r['p']<0.01 else ('*' if r['p']<0.05 else 'ns'))
                print(f'{key:<40} {mname:<15} {r["n"]:>6} {r["da"]:>8.3f} {r["p"]:>10.4f} {sig}')

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

TAU_FIXED = 1.0

def cliff_metrics(df_sub, pred_col, tau):
    y  = df_sub.is_cliff.astype(int).values
    sc = np.abs(df_sub[pred_col].values)
    yh = (sc >= tau).astype(int)
    tp = int(((y==1)&(yh==1)).sum())
    fp = int(((y==0)&(yh==1)).sum())
    tn = int(((y==0)&(yh==0)).sum())
    fn = int(((y==1)&(yh==0)).sum())
    P  = tp/(tp+fp) if tp+fp else 0.0
    R  = tp/(tp+fn) if tp+fn else 0.0
    F  = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0.0
    out = {'n': len(df_sub), 'n_cliffs': int(y.sum()), 'tau': tau,
           'tp':tp,'fp':fp,'tn':tn,'fn':fn,'precision':P,'recall':R,'f1':F}
    if 0 < y.sum() < len(y):
        out['ap'] = float(average_precision_score(y, sc))
        out['roc_auc'] = float(roc_auc_score(y, sc))
    return out

def bootstrap_ap_diff(df_sub, pred_col='pred_rel_mean', base_col='pred_knn_delta', nboot=2000):
    from collections import defaultdict
    y  = df_sub.is_cliff.astype(int).values
    sr = np.abs(df_sub[pred_col].values)
    sk = np.abs(df_sub[base_col].values)
    cl_map = defaultdict(list)
    for i, c in enumerate(df_sub['smiles_1'].values):
        cl_map[c].append(i)
    keys = list(cl_map)
    rng  = np.random.RandomState(0)
    diffs = []
    for _ in range(nboot):
        pick = rng.randint(0, len(keys), len(keys))
        idx  = np.array([j for p in pick for j in cl_map[keys[p]]], dtype=int)
        yt   = y[idx]
        if yt.sum()==0 or yt.sum()==len(yt): continue
        diffs.append(average_precision_score(yt,sr[idx]) - average_precision_score(yt,sk[idx]))
    if len(diffs) < 20: return None, None, len(diffs)
    return float(np.percentile(diffs,2.5)), float(np.percentile(diffs,97.5)), len(diffs)

cliff_results = {}
print(f'\nCliff detection (tan>={CLIFF_SIM}, |Δ|>={CLIFF_DELTA}, tau={TAU_FIXED})')
for pt in ['test_test', 'test_train']:
    sub = df_eval[(df_eval.pair_type==pt) & (df_eval.tanimoto>=CLIFF_SIM)]
    if len(sub) == 0: continue
    n_cl = sub.is_cliff.sum()
    print(f'\n── {pt}  close={len(sub):,}  cliffs={n_cl:,}  prev={n_cl/len(sub):.1%}')
    for mname, col in [('rel_siamese','pred_rel_mean'), ('knn_5nn','pred_knn_delta')]:
        m = cliff_metrics(sub, col, TAU_FIXED)
        cliff_results[f'{pt}|{mname}'] = m
        ap_s = f"{m.get('ap',float('nan')):.3f}" if 'ap' in m else 'N/A'
        print(f'  {mname:<20} AP={ap_s}  F1={m["f1"]:.3f}  P={m["precision"]:.3f}  R={m["recall"]:.3f}  tp={m["tp"]} fn={m["fn"]}')
    if n_cl >= 5:
        lo, hi, nv = bootstrap_ap_diff(sub)
        if lo is not None:
            sig = '✓ excludes 0' if lo>0 else ('✗ crosses 0' if hi>0 else '✗ negative')
            print(f'  ΔAP(rel−knn) CI₉₅=[{lo:.3f},{hi:.3f}]  ({nv} resamples)  {sig}')

In [ ]:
from scipy.stats import spearmanr, wilcoxon

print('Computing reference-anchored predictions for test molecules...')
df_test_mols  = df_all[df_all.split=='test'].reset_index(drop=True)
df_train_mols = df_all[df_all.split=='train']
by_ser_train  = df_train_mols.groupby('series')

test_anch = np.full((len(scorer.models), len(df_test_mols)), np.nan)
for k, row in df_test_mols.iterrows():
    if row.series not in by_ser_train.groups: continue
    refs_df = by_ser_train.get_group(row.series)
    ref_smis, ref_y = list(refs_df.smiles), refs_df.pIC50.values.astype(float)
    qg = [get_graph(row.smiles)] * len(ref_smis)
    rg = [get_graph(s) for s in ref_smis]
    ok = [i for i,(a,b) in enumerate(zip(qg,rg)) if a is not None and b is not None]
    if not ok: continue
    for mi, mod in enumerate(scorer.models):
        raw = scorer._raw(mod, [qg[i] for i in ok], [rg[i] for i in ok])
        test_anch[mi,k] = float(np.mean(ref_y[ok] + raw))

anch_mean     = np.nanmean(test_anch, axis=0)
test_knn_abs  = knn_pred_all[~is_train_a]

spearman_rel, spearman_knn = [], []
for series, g in df_test_mols.groupby('series'):
    if len(g) < 5: continue
    idx = g.index.values
    yt  = g.pIC50.values
    as_ = anch_mean[idx]
    if not np.isfinite(as_).all(): continue
    spearman_rel.append(float(spearmanr(yt, as_)[0]))
    spearman_knn.append(float(spearmanr(yt, test_knn_abs[idx])[0]))

print(f'\n=== Within-Series Spearman ρ (series with ≥5 test members) ===')
print(f'  Series: {len(spearman_rel)}')
print(f'  rel_siamese: mean ρ = {np.mean(spearman_rel):.4f}')
print(f'  knn_5nn    : mean ρ = {np.mean(spearman_knn):.4f}')
if len(spearman_rel) >= 6:
    try:
        _, p_w = wilcoxon(spearman_rel, spearman_knn)
        diff = np.mean(np.array(spearman_rel) - np.array(spearman_knn))
        print(f'  Wilcoxon: Δρ={diff:+.4f}  p={p_w:.4f}  {"sig" if p_w<0.05 else "ns"}')
    except Exception as e:
        print(f'  Wilcoxon: {e}')

## 6 · Comparison Table

In [ ]:
print('=== Prototype (1,102 mol) vs MMP Dataset (1,924 mol) ===')
print(f'{"Metric":<45} {"Original":>12} {"MMP Run":>14}')
print('─'*75)

mmp_ap_tt  = cliff_results.get('test_test|rel_siamese',{}).get('ap', float('nan'))
mmp_ap_knn = cliff_results.get('test_test|knn_5nn',{}).get('ap', float('nan'))
n_cl_tt    = cliff_results.get('test_test|rel_siamese',{}).get('n_cliffs', 'N/A')

rows = [
    ('Total molecules',              '1,102',     f'{len(df_all):,}'),
    ('Train / test',                 '961 / 141', f'{n_train:,} / {n_test:,}'),
    ('MMP training pairs',           '4,578',     f'{len(train_dataset):,}'),
    ('Within-series ρ (rel)',        '0.581',     f'{np.mean(spearman_rel):.3f}' if spearman_rel else 'N/A'),
    ('Within-series ρ (knn)',        '~0.19',     f'{np.mean(spearman_knn):.3f}' if spearman_knn else 'N/A'),
    ('Test-test cliff positives',    '5',         str(n_cl_tt)),
    ('AP test-test (rel_siamese)',   '0.731',     f'{mmp_ap_tt:.3f}' if np.isfinite(mmp_ap_tt) else 'N/A'),
    ('AP test-test (knn_5nn)',       '0.331',     f'{mmp_ap_knn:.3f}' if np.isfinite(mmp_ap_knn) else 'N/A'),
]
for name, o, m in rows:
    print(f'{name:<45} {str(o):>12} {str(m):>14}')

## 7 · Save & Download Results

In [ ]:
import json

def safe_json(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(f'{type(o)}')

summary = {
    'dataset': 'Dablander MMP Moonshot Mpro (1,924 molecules, 12,594 pairs)',
    'split': {'n_train': n_train, 'n_test': n_test, 'n_train_mmp_pairs': len(train_dataset)},
    'cliff_definition': {'tanimoto': CLIFF_SIM, 'abs_delta_pIC50': CLIFF_DELTA},
    'within_series_spearman': {
        'n_series': len(spearman_rel),
        'rel_mean': float(np.mean(spearman_rel)) if spearman_rel else None,
        'knn_mean': float(np.mean(spearman_knn)) if spearman_knn else None,
        'per_series_rel': spearman_rel,
        'per_series_knn': spearman_knn,
    },
    'cliff_detection': cliff_results,
    'direction_accuracy': dir_results,
    'training_epochs': EPOCHS,
    'checkpoints': ckpt_paths,
}

with open('/content/results_mmp/summary_mmp.json', 'w') as f:
    json.dump(summary, f, indent=2, default=safe_json)

print('Results saved:')
for fn in sorted(os.listdir('/content/results_mmp')):
    print(f"  {fn}: {os.path.getsize(f'/content/results_mmp/{fn}'):,} bytes")

In [ ]:
!zip -r /content/cliffforge_mmp_results.zip /content/results_mmp/ /content/mmp_models/

from google.colab import files
files.download('/content/cliffforge_mmp_results.zip')
print('Download triggered.')

---
## Done

**Outputs:**
- `/content/mmp_data/processed/` — curated molecule + MMP pair tables
- `/content/mmp_models/rel_siamese_mmp_s{0,1,2}.pt` — trained checkpoints
- `/content/results_mmp/mmp_test_predictions.csv` — per-pair predictions
- `/content/results_mmp/summary_mmp.json` — all metrics in JSON